# Arabic Tokenizers Benchmark Sweep — Full Analysis

**Experiment**: LightEval benchmark sweep (ACVA · AlGhafa · Culture-Arabic-MMLU · Arabic-Exam)  
**Model**: LLaMA 3.2-1B (`meta-llama/Llama-3.2-1B`)  
**Tokenizers**: BPE · WordPiece · MorphoBPE × {16K, 32K, 50K} vocab · CharacterBERT · char-JABER  
**Total experiments**: 44 (36 subword + 8 character-level)  

> CharacterBERT and char-JABER previously failed due to a CUDA device-placement bug.
> That has been fixed; all 44 experiments now have valid accuracy scores.

---

## 0 · Imports and Setup

In [ ]:
import json, pathlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import seaborn as sns

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 130, 'font.size': 10})
sns.set_theme(style='whitegrid')

REPORT_PATH = pathlib.Path('../outputs/experiments/benchmark_sweep/comparison_report.json')

TASKS = ['acva', 'alghafa', 'culture_arabic_mmlu', 'arabic_exam']
TASK_LABELS = {
    'acva': 'ACVA', 'alghafa': 'AlGhafa',
    'culture_arabic_mmlu': 'Culture-MMLU', 'arabic_exam': 'Arabic-Exam',
}
# ACVA is binary (صح / خطأ) → random baseline 0.50; all others are 4-way MCQ → 0.25
RANDOM_BASELINE = {
    'acva': 0.50, 'alghafa': 0.25, 'culture_arabic_mmlu': 0.25, 'arabic_exam': 0.25,
}

TOK_ORDER  = ['bpe', 'wordpiece', 'morpho_bpe', 'character_bert', 'char_jaber']
TOK_LABELS = {
    'bpe': 'BPE', 'wordpiece': 'WordPiece', 'morpho_bpe': 'MorphoBPE',
    'character_bert': 'CharacterBERT', 'char_jaber': 'char-JABER',
}
TOK_COLORS = {
    'bpe':            '#2196F3',
    'wordpiece':      '#FF9800',
    'morpho_bpe':     '#4CAF50',
    'character_bert': '#9C27B0',
    'char_jaber':     '#F44336',
}
SUBWORD_TOKS = ['bpe', 'wordpiece', 'morpho_bpe']
CHAR_TOKS    = ['character_bert', 'char_jaber']
VOCAB_MARKERS = {16000: 'o', 32000: 's', 50000: '^'}

with open(REPORT_PATH) as f:
    raw = json.load(f)
print(f'Loaded {len(raw)} experiment entries')

## 1 · Parse into DataFrames

In [ ]:
records = []
for key, val in raw.items():
    if 'error' in val:
        continue
    cfg, intr, train, ds = val['config'], val['intrinsic'], val['training'], val['downstream']
    task  = cfg['task']
    acc   = ds.get(task, {}).get('accuracy')
    nsamp = ds.get(task, {}).get('num_samples')
    vs    = cfg['vocab_size']   # None for character-level tokenizers

    records.append({
        'key':               key,
        'tokenizer':         cfg['tokenizer'],
        'vocab_size':        vs,
        'vocab_size_actual': intr['vocab_size'],
        'vocab_label':       (f"{int(vs // 1000)}K" if vs else '—'),
        'task':              task,
        'fertility':         intr['fertility'],
        'compression_ratio': intr['compression_ratio'],
        'unk_rate':          intr['unk_rate'],
        'vocab_coverage':    intr['vocab_coverage'],
        'avg_token_count':   intr['avg_token_count'],
        'train_loss':        train['train_loss'],
        'total_steps':       train['total_steps'],
        'training_time_s':   train['training_time_seconds'],
        'epochs':            train['epochs_completed'],
        'accuracy':          acc,
        'num_samples':       nsamp,
    })

df = pd.DataFrame(records)
df['tokenizer'] = pd.Categorical(df['tokenizer'], TOK_ORDER)
df = df.sort_values(['tokenizer', 'vocab_size', 'task']).reset_index(drop=True)
df['is_subword'] = df['tokenizer'].isin(SUBWORD_TOKS)
df['tok_label']  = df['tokenizer'].map(TOK_LABELS)
df['config_key'] = df.apply(
    lambda r: r.tok_label if r.vocab_label == '—' else f"{r.tok_label} {r.vocab_label}",
    axis=1
)

df_sub  = df[df.is_subword].copy()
df_char = df[~df.is_subword].copy()

print(f"Total experiments : {len(df)}")
print(f"  Subword  (36)   : BPE / WordPiece / MorphoBPE × 3 vocab sizes × 4 tasks")
print(f"  Char-level (8)  : CharacterBERT / char-JABER × 4 tasks")
df.head(8)

## 2 · Summary Table

In [ ]:
agg_spec = dict(
    vocab_size_actual = ('vocab_size_actual', 'first'),
    fertility         = ('fertility',         'first'),
    compression_ratio = ('compression_ratio', 'first'),
    avg_token_count   = ('avg_token_count',   'first'),
    unk_rate          = ('unk_rate',           'first'),
    avg_train_loss    = ('train_loss',         'mean'),
    avg_accuracy      = ('accuracy',           'mean'),
    avg_train_time_s  = ('training_time_s',    'mean'),
)

summary = (
    df.groupby('config_key', sort=False)
    .agg(**agg_spec)
    .round(4)
    .sort_values('avg_accuracy', ascending=False)
)
print('Configuration summary — mean over 4 tasks, ranked by accuracy:')
summary.style.background_gradient(cmap='RdYlGn', subset=['avg_accuracy']).format(precision=4)

## 3 · Intrinsic Tokenizer Metrics

In [ ]:
intr_sub  = df_sub.drop_duplicates(['tokenizer', 'vocab_size']).sort_values('vocab_size')
intr_char = df_char.drop_duplicates('tokenizer')

metrics_cfg = [
    ('fertility',         'Fertility (tokens/word)'),
    ('compression_ratio', 'Compression Ratio (chars/token)'),
    ('avg_token_count',   'Avg Tokens per Text'),
    ('unk_rate',          'UNK Rate'),
]

fig, axes = plt.subplots(1, 4, figsize=(17, 4))

for ax, (metric, label) in zip(axes, metrics_cfg):
    for tok in SUBWORD_TOKS:
        sub = intr_sub[intr_sub.tokenizer == tok]
        ax.plot(sub.vocab_size / 1000, sub[metric],
                marker='o', color=TOK_COLORS[tok], label=TOK_LABELS[tok], linewidth=2)
    for tok in CHAR_TOKS:
        val = intr_char.loc[intr_char.tokenizer == tok, metric].values[0]
        ax.axhline(val, color=TOK_COLORS[tok], linestyle='--', linewidth=1.8,
                   label=f"{TOK_LABELS[tok]} ({val:.2f})", alpha=0.85)
    ax.set_title(label, fontsize=9)
    ax.set_xlabel('Vocab Size (K)')
    ax.set_xticks([16, 32, 50])
    ax.set_xlim(12, 54)

axes[0].legend(fontsize=7, loc='best')
fig.suptitle('Intrinsic Metrics: Subword (lines, x = vocab size) vs Character-level (dashed)',
             fontsize=11, y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
# Side-by-side bar comparison at 32K (subword) vs char tokenizers
intr_32k = intr_sub[intr_sub.vocab_size == 32000]
intr_cmp = pd.concat([intr_32k, intr_char], ignore_index=True)
intr_cmp['display'] = intr_cmp.apply(
    lambda r: TOK_LABELS[r.tokenizer] if r.vocab_label == '—'
              else f"{TOK_LABELS[r.tokenizer]}\n{r.vocab_label}",
    axis=1
)
colors_cmp = [TOK_COLORS[t] for t in intr_cmp.tokenizer]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (metric, label) in zip(axes, [
    ('fertility',         'Fertility ↓ (tokens/word)'),
    ('compression_ratio', 'Compression Ratio ↑ (chars/tok)'),
    ('avg_token_count',   'Avg Token Count ↓'),
]):
    bars = ax.bar(intr_cmp.display, intr_cmp[metric], color=colors_cmp, alpha=0.85, edgecolor='white')
    ax.set_title(label, fontsize=9)
    ax.tick_params(axis='x', labelsize=7.5)
    ymax = intr_cmp[metric].max()
    for bar, val in zip(bars, intr_cmp[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + ymax * 0.01,
                f'{val:.2f}', ha='center', va='bottom', fontsize=7)

fig.suptitle('All Tokenizers — Intrinsic Metrics (subword shown at 32K)', fontsize=10, y=1.03)
plt.tight_layout()
plt.show()

### Key observations — intrinsic metrics

| Tokenizer | Notable property |
|---|---|
| **char-JABER** | Fertility 5.89 — each character is its own token; sequences are 4–6× longer than subword |
| **CharacterBERT** | Fertility 1.01 (lowest) — one token per word; highest compression ratio (5.85 chars/token) |
| **CharacterBERT** | UNK rate 13.9 % — word vocabulary (50 K entries) covers only 22.6 % of unique Arabic word forms |
| **BPE** | Best compression at all vocab sizes; exact zero UNK (byte-level pre-tokenisation) |
| **MorphoBPE** | Highest fertility among subword types (≈1.9–2.0) — Farasa splits morphemes before BPE |
| **WordPiece** | Moderate fertility; near-zero UNK via `##` continuation subwords |

## 4 · Training Dynamics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 4a — train loss vs accuracy for all experiments
ax = axes[0]
for tok in TOK_ORDER:
    sub = df[df.tokenizer == tok]
    ax.scatter(sub.train_loss, sub.accuracy,
               c=TOK_COLORS[tok], label=TOK_LABELS[tok], s=40, alpha=0.8)
ax.set_xlabel('Train Loss')
ax.set_ylabel('Accuracy')
ax.set_title('Train Loss vs Accuracy (all 44 experiments)')
ax.legend(fontsize=7)

# 4b — avg train loss per tokenizer (mean over vocab sizes and tasks)
ax = axes[1]
loss_by_tok = df.groupby('tokenizer')['train_loss'].mean().reindex(TOK_ORDER)
colors_tok  = [TOK_COLORS[t] for t in TOK_ORDER]
bars = ax.bar([TOK_LABELS[t] for t in TOK_ORDER], loss_by_tok.values,
              color=colors_tok, alpha=0.85, edgecolor='white')
for bar, val in zip(bars, loss_by_tok.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
            f'{val:.3f}', ha='center', va='bottom', fontsize=7.5)
ax.set_title('Avg Train Loss by Tokenizer')
ax.set_ylabel('Train Loss')
ax.tick_params(axis='x', rotation=20, labelsize=8)

# 4c — avg training time per configuration
ax = axes[2]
time_agg = (
    df.groupby(['tokenizer', 'vocab_label'])['training_time_s']
    .mean()
    .reset_index()
    .sort_values(['tokenizer', 'vocab_label'])
)
xlabels  = [f"{TOK_LABELS[r.tokenizer]}\n{r.vocab_label}" for _, r in time_agg.iterrows()]
colors_t = [TOK_COLORS[r.tokenizer] for _, r in time_agg.iterrows()]
ax.bar(range(len(time_agg)), time_agg.training_time_s, color=colors_t, alpha=0.85, edgecolor='white')
ax.set_xticks(range(len(time_agg)))
ax.set_xticklabels(xlabels, fontsize=6.5, rotation=45)
ax.set_title('Avg Training Time per Task (s)')
ax.set_ylabel('Seconds')

plt.tight_layout()
plt.show()

## 5 · Downstream Accuracy — Per-Task Comparison

In [ ]:
# Fixed display order for bars across all subplots
config_order = [
    'BPE 16K', 'BPE 32K', 'BPE 50K',
    'WordPiece 16K', 'WordPiece 32K', 'WordPiece 50K',
    'MorphoBPE 16K', 'MorphoBPE 32K', 'MorphoBPE 50K',
    'CharacterBERT', 'char-JABER',
]
config_colors = [
    TOK_COLORS['bpe']] * 3 + [TOK_COLORS['wordpiece']] * 3 + \
    [TOK_COLORS['morpho_bpe']] * 3 + [TOK_COLORS['character_bert'], TOK_COLORS['char_jaber']
]

fig, axes = plt.subplots(2, 2, figsize=(15, 9))
axes = axes.flatten()
x = np.arange(len(config_order))

for ax, task in zip(axes, TASKS):
    sub = df[df.task == task].set_index('config_key').reindex(config_order)
    bars = ax.bar(x, sub.accuracy, color=config_colors, alpha=0.85, edgecolor='white')
    rb = RANDOM_BASELINE[task]
    ax.axhline(rb, color='red', linestyle='--', linewidth=1.2,
               label=f'Random ({rb:.2f})')
    ax.set_xticks(x)
    ax.set_xticklabels(config_order, fontsize=6.5, rotation=40, ha='right')
    ax.set_title(TASK_LABELS[task], fontsize=11)
    ax.set_ylabel('Accuracy')
    ymax = (sub.accuracy.max() or 0) + 0.08
    ax.set_ylim(0, max(ymax, rb + 0.15))
    for bar, val in zip(bars, sub.accuracy):
        if not np.isnan(val):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=5.5)
    ax.legend(fontsize=7)

legend_handles = [Patch(facecolor=TOK_COLORS[t], label=TOK_LABELS[t]) for t in TOK_ORDER]
fig.legend(handles=legend_handles, loc='lower center', ncol=5,
           fontsize=8.5, title='Tokenizer', bbox_to_anchor=(0.5, -0.02))
fig.suptitle('Downstream Accuracy per Task and Configuration', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 6 · Accuracy Heatmap — All Configurations × Tasks

In [ ]:
heat = (
    df.pivot_table(index='config_key', columns='task', values='accuracy')
    .reindex(config_order)
)
heat.columns = [TASK_LABELS[c] for c in heat.columns]
heat['Mean'] = heat.mean(axis=1)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(heat, ax=ax, annot=True, fmt='.3f', cmap='RdYlGn',
            vmin=0.24, vmax=0.62, linewidths=0.5, cbar_kws={'shrink': 0.7})
ax.set_title('Accuracy Heatmap — All Configurations × Tasks', fontsize=11)
ax.set_xlabel('')
ax.set_ylabel('')
ax.tick_params(axis='y', labelsize=8)
plt.tight_layout()
plt.show()

## 7 · Effect of Vocab Size on Accuracy (Subword Tokenizers)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=False)

for ax, task in zip(axes, TASKS):
    sub = df_sub[df_sub.task == task]
    for tok in SUBWORD_TOKS:
        s = sub[sub.tokenizer == tok].sort_values('vocab_size')
        ax.plot(s.vocab_size / 1000, s.accuracy,
                marker='o', color=TOK_COLORS[tok], label=TOK_LABELS[tok], linewidth=2)
    rb = RANDOM_BASELINE[task]
    ax.axhline(rb, color='red', linestyle='--', linewidth=1, alpha=0.7, label=f'Random ({rb:.2f})')
    ax.set_title(TASK_LABELS[task], fontsize=10)
    ax.set_xlabel('Vocab Size (K)')
    ax.set_ylabel('Accuracy')
    ax.set_xticks([16, 32, 50])

axes[0].legend(fontsize=8)
fig.suptitle('Accuracy vs Vocab Size — Subword Tokenizers', fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Accuracy delta: 50K − 16K
delta = (
    df_sub.pivot_table(index=['tokenizer', 'task'], columns='vocab_size', values='accuracy')
    .assign(delta=lambda x: x[50000] - x[16000])
    .reset_index()
)

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(TASKS))
w = 0.25
for i, tok in enumerate(SUBWORD_TOKS):
    vals = [delta[(delta.tokenizer == tok) & (delta.task == t)]['delta'].values[0] for t in TASKS]
    ax.bar(x + (i-1)*w, vals, width=w, label=TOK_LABELS[tok],
           color=TOK_COLORS[tok], alpha=0.9)

ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels([TASK_LABELS[t] for t in TASKS])
ax.set_title('Accuracy Delta: 50K vocab − 16K vocab', fontsize=11)
ax.set_ylabel('Δ Accuracy')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 8 · Character-Level vs Best Subword

In [ ]:
# Best subword config per task (max over all subword configurations)
best_sub = df_sub.groupby('task')['accuracy'].max().reset_index().rename(columns={'accuracy': 'best_subword'})

char_acc = (
    df_char.groupby(['tokenizer', 'task'])['accuracy']
    .first()
    .reset_index()
    .merge(best_sub, on='task')
)
char_acc['gap'] = char_acc['accuracy'] - char_acc['best_subword']
char_acc['task_label'] = char_acc['task'].map(TASK_LABELS)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Grouped bar: char tokenizers vs best subword per task
ax = axes[0]
x = np.arange(len(TASKS))
w = 0.25
ax.bar(x - w, best_sub.sort_values('task').best_subword, width=w,
       label='Best Subword', color='#607D8B', alpha=0.8)
for i, tok in enumerate(CHAR_TOKS):
    vals = [char_acc[(char_acc.tokenizer == tok) & (char_acc.task == t)]['accuracy'].values[0]
            for t in TASKS]
    ax.bar(x + i*w, vals, width=w, label=TOK_LABELS[tok],
           color=TOK_COLORS[tok], alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels([TASK_LABELS[t] for t in TASKS])
ax.set_title('Char-level vs Best Subword Accuracy')
ax.set_ylabel('Accuracy')
ax.legend(fontsize=8)

# Gap bar chart (positive = char outperforms best subword)
ax = axes[1]
for i, tok in enumerate(CHAR_TOKS):
    sub_gap = char_acc[char_acc.tokenizer == tok].sort_values('task')
    offsets = np.arange(len(TASKS)) + (i - 0.5) * 0.3
    colors_gap = [TOK_COLORS[tok] if g >= 0 else '#EF9A9A' for g in sub_gap.gap]
    ax.bar(offsets, sub_gap.gap, width=0.28, color=colors_gap,
           label=TOK_LABELS[tok], alpha=0.85)

ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(np.arange(len(TASKS)))
ax.set_xticklabels([TASK_LABELS[t] for t in TASKS])
ax.set_title('Accuracy Gap: Char-level − Best Subword\n(positive = char wins)')
ax.set_ylabel('Δ Accuracy')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 9 · Intrinsic–Accuracy Correlation

In [ ]:
intrinsic_feats = ['fertility', 'compression_ratio', 'avg_token_count', 'unk_rate', 'vocab_coverage']
corr_matrix = df[intrinsic_feats + ['accuracy']].corr()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Correlation heatmap
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm',
            vmin=-1, vmax=1, linewidths=0.5, ax=axes[0])
axes[0].set_title('Pearson Correlation: Intrinsic Metrics vs Accuracy', fontsize=10)

# Scatter: fertility vs accuracy
ax = axes[1]
for tok in TOK_ORDER:
    sub = df[df.tokenizer == tok]
    ax.scatter(sub.fertility, sub.accuracy,
               c=TOK_COLORS[tok], label=TOK_LABELS[tok], s=40, alpha=0.8)
m, b = np.polyfit(df.fertility, df.accuracy, 1)
xs = np.linspace(df.fertility.min(), df.fertility.max(), 100)
ax.plot(xs, m*xs + b, 'k--', linewidth=1.2, alpha=0.5)
r = df[['fertility', 'accuracy']].corr().iloc[0, 1]
ax.set_xlabel('Fertility (tokens/word)')
ax.set_ylabel('Accuracy')
ax.set_title(f'Fertility vs Accuracy  (r = {r:.3f})')
ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

## 10 · Best Performers per Task

In [ ]:
top3 = (
    df.sort_values('accuracy', ascending=False)
    .groupby('task')
    .head(3)
    [['task', 'config_key', 'accuracy', 'fertility', 'compression_ratio', 'train_loss']]
)
top3['task'] = top3['task'].map(TASK_LABELS)
top3 = top3.set_index(['task', 'config_key'])
print('Top-3 performers per task:')
top3.style.background_gradient(cmap='YlGn', subset=['accuracy']).format(precision=4)

In [ ]:
overall = (
    df.groupby('config_key', sort=False)
    .agg(mean_acc=('accuracy', 'mean'), std_acc=('accuracy', 'std'))
    .sort_values('mean_acc', ascending=False)
    .reset_index()
)
print('Overall ranking — mean accuracy across 4 tasks:')
display(overall.style.background_gradient(cmap='RdYlGn', subset=['mean_acc']).format(precision=4))

winner = overall.iloc[0]
print(f"\nWinner: {winner.config_key}  →  mean accuracy = {winner.mean_acc:.4f}")

## 11 · Cross-Task Consistency (Radar Chart)

In [ ]:
# Mean accuracy per (tokenizer, task) — averaged over vocab sizes for subword
cat_df = (
    df.groupby(['tokenizer', 'task'])['accuracy']
    .mean()
    .reset_index()
    .pivot(index='tokenizer', columns='task', values='accuracy')
    .reindex(TOK_ORDER)
)
cat_df.columns = [TASK_LABELS[c] for c in cat_df.columns]

tasks_r = list(cat_df.columns)
N = len(tasks_r)
angles = [n / float(N) * 2 * np.pi for n in range(N)] + [0]

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw={'polar': True})
for tok in TOK_ORDER:
    vals = cat_df.loc[tok].tolist() + [cat_df.loc[tok].tolist()[0]]
    ax.plot(angles, vals, color=TOK_COLORS[tok], linewidth=2, label=TOK_LABELS[tok])
    ax.fill(angles, vals, color=TOK_COLORS[tok], alpha=0.07)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(tasks_r, fontsize=9)
ax.set_title('Cross-Task Accuracy (avg over vocab sizes)', fontsize=11, pad=15)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
consistency = (
    df.groupby('config_key', sort=False)['accuracy']
    .agg(['mean', 'std', 'min', 'max'])
    .rename(columns={'mean': 'avg_acc', 'std': 'std_acc', 'min': 'min_acc', 'max': 'max_acc'})
    .round(4)
    .sort_values('avg_acc', ascending=False)
)
print('Accuracy consistency across 4 tasks per configuration:')
consistency.style.background_gradient(cmap='RdYlGn', subset=['avg_acc']).format(precision=4)

## 12 · Efficiency — Accuracy per Training Second

In [ ]:
df['acc_per_sec'] = df['accuracy'] / df['training_time_s']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter: training time vs accuracy
ax = axes[0]
for tok in TOK_ORDER:
    sub = df[df.tokenizer == tok]
    ax.scatter(sub.training_time_s, sub.accuracy,
               s=55, c=TOK_COLORS[tok], label=TOK_LABELS[tok], alpha=0.75, edgecolors='white')
ax.set_xlabel('Training Time (seconds)')
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy vs Training Time')
ax.legend(fontsize=7)

# Bar: efficiency (accuracy per second × 10³)
ax = axes[1]
eff = (
    df.groupby('config_key', sort=False)['acc_per_sec']
    .mean()
    .reindex(config_order)
    .reset_index()
)
colors_eff = []
for k in eff.config_key:
    tok_name = next(t for t in TOK_ORDER if TOK_LABELS[t] in k)
    colors_eff.append(TOK_COLORS[tok_name])

ax.bar(range(len(eff)), eff.acc_per_sec * 1000, color=colors_eff, alpha=0.85, edgecolor='white')
ax.set_xticks(range(len(eff)))
ax.set_xticklabels(eff.config_key, fontsize=6.5, rotation=45, ha='right')
ax.set_title('Efficiency: Accuracy × 10³ / Training Second')
ax.set_ylabel('Acc × 10³ / sec')

plt.tight_layout()
plt.show()

## 13 · Full Results Table

In [ ]:
full = (
    df[[
        'config_key', 'task',
        'fertility', 'compression_ratio', 'avg_token_count',
        'train_loss', 'training_time_s',
        'accuracy', 'num_samples',
    ]]
    .copy()
    .rename(columns={
        'config_key':       'configuration',
        'compression_ratio':'comp_ratio',
        'avg_token_count':  'avg_tokens',
        'training_time_s':  'train_sec',
    })
)
full['task'] = full['task'].map(TASK_LABELS)
full = full.set_index(['configuration', 'task'])

pd.set_option('display.max_rows', 50)
full.style \
    .background_gradient(cmap='RdYlGn', subset=['accuracy']) \
    .background_gradient(cmap='Blues_r', subset=['train_loss']) \
    .format(precision=4)

## 14 · Key Findings and Conclusions

### Overall ranking (mean accuracy across all 4 tasks)

char-JABER and BPE 50K tie at the top; CharacterBERT outperforms MorphoBPE and WordPiece.

### Downstream performance

| Finding | Evidence |
|---|---|
| **BPE dominates on ACVA** | BPE 32K scores 0.599 vs WordPiece best 0.465, MorphoBPE best 0.594 |
| **char-JABER is the best on Arabic-Exam** | 0.3156 vs BPE best 0.3010, MorphoBPE best 0.3101 |
| **char-JABER matches BPE on ACVA and AlGhafa** | 0.5952 / 0.5803 vs BPE 0.5990 / 0.5805 |
| **CharacterBERT is competitive on binary ACVA** | 0.5909 — only 0.008 behind best BPE |
| **CharacterBERT struggles on harder MCQ** | Culture-MMLU 0.2476, Arabic-Exam 0.2675 (near random) |
| **Culture-MMLU is uniformly hard** | All tokenizers score 0.244–0.270 — close to 4-way random (0.25) |
| **WordPiece consistently underperforms** | Lowest avg accuracy in all 9 subword configurations |
| **MorphoBPE 32K/50K best among morphological** | Competitive on Arabic-Exam (0.310/0.306) despite high fertility |

### ACVA baseline correction

ACVA is a **binary** (True/False) benchmark — random baseline is **0.50**, not 0.25.  
All tokenizers score only 0.59–0.60 on ACVA, which is a modest 9–10 pp above chance,
not the large margin it appears when the 0.25 baseline is incorrectly applied.

### Vocab size effects (subword only)

| Finding | Evidence |
|---|---|
| **Larger vocab helps BPE slightly** | 16K→50K: +0.0015 on ACVA, +0.0 to −0.01 on others |
| **WordPiece accuracy is non-monotone** | Peaks at 32K on AlGhafa (0.548), drops at 50K (0.516) |
| **MorphoBPE insensitive to vocab size** | ≤0.004 spread at any given task |

### Intrinsic metrics

- Fertility shows **no clear linear relationship** with accuracy across all 5 tokenizer types —
  char-JABER (fertility 5.89) matches BPE (fertility 1.3–1.5) despite 4× longer sequences.
- CharacterBERT's 13.9 % UNK rate (word vocab) does **not** severely hurt accuracy because
  the CharCNN input always uses full character sequences; UNK affects the output prediction head only.

### Recommendations

1. **BPE 32K** — best balance of accuracy, training speed, and compression for general use.
2. **char-JABER** — surprisingly competitive overall; strongest on Arabic-Exam; worth deeper investigation
   (sequence length 4–6× longer, but train loss is the lowest of all tokenizers).
3. **Avoid WordPiece** — consistently the weakest across all benchmarks and vocab sizes.
4. **Culture-MMLU needs more SFT data** — 10 % split (≈1.3K examples) may be insufficient
   for a 4-way benchmark this hard; consider increasing `train_split_ratio` to 0.20.
5. **CharacterBERT is promising for simple tasks** but needs a larger word vocabulary or
   a hybrid OOV strategy to compete on harder multi-class benchmarks.